# Qwen3.5-2B Exam Generator — Unsloth Fine-Tuning & GGUF Export

This notebook distills **Exam Generator** domain expertise (`<<<STUDENT_SHEET>>>` + `<<<ANSWER_KEY>>>` structured exam synthesis with misconception-driven distractors, Bloom-aligned stems, and solve-from-scratch answer keys) into **`unsloth/Qwen3.5-2B`**.

### Configuration Summary
- **Base Model:** `unsloth/Qwen3.5-2B` (`load_in_4bit=False`, bf16/fp16 LoRA — ~5 GB VRAM on T4)
- **Context Length:** `max_seq_length = 3072`
- **Target Hardware:** Google Colab Free **T4 (15 GB VRAM)**, also compatible with **L4 / A100**
- **Training Method:** `SFTTrainer` + `SFTConfig` + `train_on_responses_only` (`enable_thinking=False`)
- **Exports:** LoRA adapter (`qwen3.5_2b_exam_lora`), `Q4_K_M` GGUF (`qwen3.5_2b_exam_gguf`), and Ollama `Modelfile` (ready for local **RTX 3050 4GB** or **RTX 5060 Ti 16GB**)

## Step 1 — Setup & Install

Installs modern **Unsloth** with Qwen3.5-compatible `transformers >= 5.2.0` and `trl >= 0.22.2` using `uv` / `pip`, then verifies GPU availability, VRAM, and `bf16` support.

In [ ]:
import importlib.util
import os
import shutil
import subprocess
import sys


def run_cmd(cmd: list[str]) -> None:
    print("+", " ".join(cmd))
    subprocess.check_call(cmd)


if "COLAB_" in "".join(os.environ.keys()) and shutil.which("nvidia-smi") is None:
    raise RuntimeError(
        "No NVIDIA GPU attached to this Colab runtime! "
        "Click Runtime -> Change runtime type -> Hardware accelerator: T4 GPU -> Save, then re-run."
    )

run_cmd([sys.executable, "-m", "pip", "install", "-q", "uv"])

if "COLAB_" in "".join(os.environ.keys()):
    # 1. Verify preinstalled PyTorch/TorchVision/TorchAudio match the same CUDA build
    torch_check = subprocess.run(
        [sys.executable, "-c", "import torch, torchvision, torchaudio"],
        capture_output=True,
        text=True,
    )
    repaired_torch = False
    if torch_check.returncode != 0:
        print("Detected PyTorch/TorchAudio CUDA mismatch — aligning wheels...")
        repaired_torch = True
        ok = subprocess.run([
            "uv", "pip", "install", "--system", "-q", "--force-reinstall",
            "--torch-backend=auto", "torch", "torchvision", "torchaudio",
        ]).returncode == 0
        if (
            not ok
            or subprocess.run(
                [sys.executable, "-c", "import torch, torchvision, torchaudio"],
                capture_output=True,
            ).returncode != 0
        ):
            run_cmd([
                "uv", "pip", "install", "--system", "-q", "--force-reinstall",
                "--index-url", "https://download.pytorch.org/whl/cu130",
                "torch", "torchvision", "torchaudio",
            ])

    # 2. Install/upgrade non-PyTorch deps normally (never triggers a partial torch upgrade)
    run_cmd([
        "uv", "pip", "install", "--system", "-q", "--upgrade",
        "sentencepiece", "protobuf", "datasets>=3.4.1,<4.4.0",
        "huggingface_hub>=1.3.0", "tokenizers>=0.22.2", "safetensors",
        "hf_transfer", "tyro", "msgspec", "httpx", "einops",
        "regex", "pyyaml", "packaging", "psutil", "tqdm", "filelock", "typer-slim",
    ])

    # 3. Install/upgrade PyTorch-bound packages with --no-deps so torch stays aligned with torchaudio
    torch_deps = [
        "bitsandbytes", "accelerate", "peft", "cut_cross_entropy",
        "unsloth_zoo", "unsloth", "transformers>=5.2.0", "trl>=0.22.2",
    ]
    if importlib.util.find_spec("triton") is None:
        torch_deps.append("triton")
    run_cmd(["uv", "pip", "install", "--system", "-q", "--upgrade", "--no-deps", *torch_deps])

    if repaired_torch and "torch" in sys.modules:
        print("\n[ACTION REQUIRED] PyTorch was already loaded in memory — click Runtime > Restart session, then re-run.")
else:
    uv_install = ["uv", "pip", "install", "-q"]
    if sys.prefix == sys.base_prefix:
        uv_install.append("--system")
    run_cmd([
        *uv_install,
        "unsloth", "transformers>=5.2.0", "trl>=0.22.2",
        "datasets>=3.4.1", "sentencepiece", "protobuf", "hf_transfer",
    ])

In [ ]:
import torch
import transformers
import trl
import peft
import unsloth

print("=== Package Versions ===")
print(f"PyTorch      : {torch.__version__}")
print(f"Transformers : {transformers.__version__}")
print(f"TRL          : {trl.__version__}")
print(f"PEFT         : {peft.__version__}")
print(f"Unsloth      : {getattr(unsloth, '__version__', 'installed')}")

print("\n=== GPU & Precision Check ===")
assert torch.cuda.is_available(), (
    "CUDA GPU is required for fine-tuning. "
    "In Google Colab: click Runtime -> Change runtime type -> T4 GPU -> Save, then re-run Step 1 & 2."
)
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader
gpu_props = torch.cuda.get_device_properties(0)
vram_gb = gpu_props.total_memory / (1024 ** 3)
bf16_ok = torch.cuda.is_bf16_supported()
print(f"Active GPU   : {gpu_props.name}")
print(f"Total VRAM   : {vram_gb:.2f} GB")
print(f"BF16 Support : {bf16_ok} (using {'bf16' if bf16_ok else 'fp16'})")

## Step 2 — Load `unsloth/Qwen3.5-2B` (bf16/fp16 LoRA)

Per Unsloth's official Qwen3.5 recommendation, we set `load_in_4bit=False` to avoid 4-bit QLoRA quantization variance. At 2B parameters in fp16/bf16 LoRA, the model uses roughly ~5 GB VRAM, leaving generous headroom on a 15 GB T4 for `max_seq_length = 3072`.

In [ ]:
import torch
from unsloth import FastLanguageModel

MODEL_NAME = "unsloth/Qwen3.5-2B"
max_seq_length = 3072
dtype = None          # Auto-detect: fp16 on T4, bf16 on L4/A100
load_in_4bit = False  # Avoid 4-bit quantization variance on Qwen3.5-2B (~5 GB VRAM)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)

model.print_trainable_parameters()
allocated_gb = torch.cuda.memory_allocated() / (1024 ** 3)
reserved_gb = torch.cuda.memory_reserved() / (1024 ** 3)
print(f"VRAM allocated: {allocated_gb:.2f} GB | reserved: {reserved_gb:.2f} GB")

## Step 3 — Load & Format `dataset_train.jsonl` and `dataset_val.jsonl`

Supports three ways to supply the dataset:
1. Place `dataset_train.jsonl` and `dataset_val.jsonl` in the working directory or Google Drive (`DRIVE_DATA_DIR`).
2. Supply `dataset_exam_generator.jsonl` (auto-splits 90% train / 10% val with `seed=3407`).
3. Upload interactively via Colab's file picker if none of the files are present yet.

In [ ]:
import sys
from pathlib import Path
from datasets import load_dataset

# Optional: set USE_GOOGLE_DRIVE = True if your JSONL files live in Google Drive
USE_GOOGLE_DRIVE = False
DRIVE_DATA_DIR = Path("/content/drive/MyDrive/qwen_exam_generator")

if USE_GOOGLE_DRIVE and "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")


def _find_file(filename: str) -> Path | None:
    candidates = [
        Path(filename),
        Path("/content") / filename,
        DRIVE_DATA_DIR / filename,
    ]
    for path in candidates:
        if path.is_file():
            return path
    return None


train_file = _find_file("dataset_train.jsonl")
val_file = _find_file("dataset_val.jsonl")
combined_file = _find_file("dataset_exam_generator.jsonl")

if (train_file is None or val_file is None) and combined_file is None and "google.colab" in sys.modules:
    from google.colab import files
    print("Upload dataset_train.jsonl & dataset_val.jsonl (or dataset_exam_generator.jsonl):")
    uploaded = files.upload()
    train_file = _find_file("dataset_train.jsonl")
    val_file = _find_file("dataset_val.jsonl")
    combined_file = _find_file("dataset_exam_generator.jsonl")

if train_file is not None and val_file is not None:
    print(f"Loading train={train_file} and val={val_file}")
    raw_ds = load_dataset(
        "json",
        data_files={"train": str(train_file), "validation": str(val_file)},
    )
    train_raw = raw_ds["train"]
    val_raw = raw_ds["validation"]
elif combined_file is not None:
    print(f"Loading {combined_file} with automatic 90/10 train/val split...")
    full_ds = load_dataset("json", data_files=str(combined_file), split="train")
    split_ds = full_ds.train_test_split(test_size=0.1, seed=3407)
    train_raw = split_ds["train"]
    val_raw = split_ds["test"]
else:
    raise FileNotFoundError(
        "Dataset not found. Provide dataset_train.jsonl + dataset_val.jsonl "
        "(or dataset_exam_generator.jsonl) in the current directory or DRIVE_DATA_DIR."
    )

print(f"Raw split sizes -> train: {len(train_raw)}, val: {len(val_raw)}")

In [ ]:
# Format conversations using Qwen3.5 ChatML template with enable_thinking=False
raw_tokenizer = getattr(tokenizer, "tokenizer", tokenizer)


def _extract_messages(example: dict) -> list[dict]:
    if "messages" in example and example["messages"]:
        return example["messages"]
    if "conversations" in example and example["conversations"]:
        role_map = {"human": "user", "gpt": "assistant", "system": "system"}
        return [
            {
                "role": role_map.get(turn.get("from", turn.get("role", "")), turn.get("role", "user")),
                "content": turn.get("value", turn.get("content", "")),
            }
            for turn in example["conversations"]
        ]
    raise KeyError("Record must contain a 'messages' (or 'conversations') field.")


def apply_chat_template_safe(
    convo: list[dict],
    add_generation_prompt: bool = False,
    enable_thinking: bool = False,
) -> str:
    try:
        return tokenizer.apply_chat_template(
            convo,
            tokenize=False,
            add_generation_prompt=add_generation_prompt,
            enable_thinking=enable_thinking,
        )
    except TypeError:
        return tokenizer.apply_chat_template(
            convo,
            tokenize=False,
            add_generation_prompt=add_generation_prompt,
        )


def formatting_prompts_func(examples: dict) -> dict:
    keys = list(examples.keys())
    n_rows = len(examples[keys[0]])
    texts = []
    for i in range(n_rows):
        row = {k: examples[k][i] for k in keys}
        convo = _extract_messages(row)
        texts.append(
            apply_chat_template_safe(
                convo,
                add_generation_prompt=False,
                enable_thinking=False,
            )
        )
    return {"text": texts}


train_ds = train_raw.map(formatting_prompts_func, batched=True)
val_ds = val_raw.map(formatting_prompts_func, batched=True)


def print_length_stats(name: str, ds) -> None:
    lengths = [len(raw_tokenizer.encode(t, add_special_tokens=False)) for t in ds["text"]]
    over_limit = sum(1 for ln in lengths if ln > max_seq_length)
    avg_len = sum(lengths) / max(1, len(lengths))
    print(
        f"[{name}] count={len(lengths)} | min={min(lengths)} | "
        f"avg={avg_len:.1f} | max={max(lengths)} | over_{max_seq_length}={over_limit}"
    )


print_length_stats("train", train_ds)
print_length_stats("val", val_ds)
print("\nPreview repr(train_ds[0]['text'][:600]):")
print(repr(train_ds[0]["text"][:600]))

## Step 4 — Train with `SFTTrainer` + `SFTConfig` + `train_on_responses_only`

We dynamically inspect `train_ds[0]["text"]` to match the exact assistant prefix (whether `<|im_start|>assistant\n<think>\n\n</think>\n\n` or `<|im_start|>assistant\n`) so `train_on_responses_only` never masks out all labels, and we verify non-empty supervised tokens before training starts.

In [ ]:
import inspect
import re
import time
import torch
from trl import SFTConfig, SFTTrainer
from unsloth.chat_templates import train_on_responses_only

# Dynamically detect exact instruction and response delimiters from formatted text
sample_text = train_ds[0]["text"]
instruction_part = "<|im_start|>user\n"

resp_match = re.search(
    r"<\|im_start\|>assistant\n(?:<think>\s*</think>\s*)?",
    sample_text,
)
if resp_match:
    response_part = resp_match.group(0)
else:
    response_part = "<|im_start|>assistant\n"

print(f"Detected instruction_part : {repr(instruction_part)}")
print(f"Detected response_part    : {repr(response_part)}")

use_bf16 = torch.cuda.is_bf16_supported()

# Build SFTConfig & SFTTrainer kwargs safely across TRL versions
sft_config_params = inspect.signature(SFTConfig.__init__).parameters
sft_trainer_params = inspect.signature(SFTTrainer.__init__).parameters

cfg_kwargs = {
    "output_dir": "outputs",
    "per_device_train_batch_size": 2,
    "per_device_eval_batch_size": 2,
    "gradient_accumulation_steps": 4,
    "num_train_epochs": 3,
    "learning_rate": 2e-4,
    "lr_scheduler_type": "cosine",
    "warmup_steps": 5,
    "eval_steps": 10,
    "save_strategy": "steps",
    "save_steps": 20,
    "save_total_limit": 2,
    "logging_steps": 1,
    "optim": "adamw_8bit",
    "weight_decay": 0.01,
    "fp16": not use_bf16,
    "bf16": use_bf16,
    "seed": 3407,
    "report_to": "none",
    "load_best_model_at_end": True,
    "metric_for_best_model": "eval_loss",
}

if "eval_strategy" in sft_config_params:
    cfg_kwargs["eval_strategy"] = "steps"
elif "evaluation_strategy" in sft_config_params:
    cfg_kwargs["evaluation_strategy"] = "steps"

if "max_length" in sft_config_params:
    cfg_kwargs["max_length"] = max_seq_length
elif "max_seq_length" in sft_config_params:
    cfg_kwargs["max_seq_length"] = max_seq_length

if "dataset_text_field" in sft_config_params:
    cfg_kwargs["dataset_text_field"] = "text"
if "dataset_num_proc" in sft_config_params:
    cfg_kwargs["dataset_num_proc"] = 2
if "packing" in sft_config_params:
    cfg_kwargs["packing"] = False

sft_config = SFTConfig(**cfg_kwargs)

trainer_kwargs = {
    "model": model,
    "train_dataset": train_ds,
    "eval_dataset": val_ds,
    "args": sft_config,
}

if "processing_class" in sft_trainer_params:
    trainer_kwargs["processing_class"] = tokenizer
elif "tokenizer" in sft_trainer_params:
    trainer_kwargs["tokenizer"] = tokenizer

if "dataset_text_field" in sft_trainer_params and "dataset_text_field" not in cfg_kwargs:
    trainer_kwargs["dataset_text_field"] = "text"
if "max_seq_length" in sft_trainer_params and "max_seq_length" not in cfg_kwargs and "max_length" not in cfg_kwargs:
    trainer_kwargs["max_seq_length"] = max_seq_length
if "dataset_num_proc" in sft_trainer_params and "dataset_num_proc" not in cfg_kwargs:
    trainer_kwargs["dataset_num_proc"] = 2
if "packing" in sft_trainer_params and "packing" not in cfg_kwargs:
    trainer_kwargs["packing"] = False

trainer = SFTTrainer(**trainer_kwargs)
trainer = train_on_responses_only(
    trainer,
    instruction_part=instruction_part,
    response_part=response_part,
)

# Critical pre-training verification: ensure response labels are NOT all masked (-100)
check_count = min(5, len(trainer.train_dataset))
for idx in range(check_count):
    sample_row = trainer.train_dataset[idx]
    labels = sample_row["labels"]
    unmasked = sum(1 for x in labels if x != -100)
    masked = sum(1 for x in labels if x == -100)
    assert unmasked > 0, (
        f"All labels were masked (-100) in row {idx}! "
        f"Check response_part={repr(response_part)} against formatted text."
    )
    if idx == 0:
        print(f"Label check row 0: total={len(labels)} | masked(-100)={masked} | supervised={unmasked}")
        unmasked_ids = [tok for tok, lbl in zip(sample_row["input_ids"], labels) if lbl != -100]
        print("Supervised target preview:")
        print(repr(raw_tokenizer.decode(unmasked_ids[:80])))

In [ ]:
# Run SFT training and evaluate on validation set
start_gpu_mem = round(torch.cuda.max_memory_reserved() / (1024 ** 3), 3)
total_gpu_mem = round(torch.cuda.get_device_properties(0).total_memory / (1024 ** 3), 3)

t0 = time.time()
trainer_stats = trainer.train()
train_elapsed = time.time() - t0

eval_metrics = trainer.evaluate()
peak_gpu_mem = round(torch.cuda.max_memory_reserved() / (1024 ** 3), 3)

print("\n=== Training & Evaluation Summary ===")
print(f"Train runtime    : {train_elapsed:.1f}s ({trainer_stats.metrics.get('train_runtime', train_elapsed):.1f}s)")
print(f"Final train loss : {trainer_stats.training_loss:.4f}")
print(f"Eval loss        : {eval_metrics.get('eval_loss', float('nan')):.4f}")
print(f"VRAM start       : {start_gpu_mem:.2f} GB / {total_gpu_mem:.2f} GB")
print(f"VRAM peak        : {peak_gpu_mem:.2f} GB / {total_gpu_mem:.2f} GB ({peak_gpu_mem / total_gpu_mem * 100:.1f}%)")

## Step 5 — Inference Verification (Aligned with `schema.py`)

Tests the fine-tuned model using the exact `CANONICAL_SYSTEM_PROMPT` from `schema.py`, Qwen3.5 non-thinking sampling (`enable_thinking=False`, `temperature=0.6`, `top_p=0.85`, `top_k=20`, `repetition_penalty=1.05`), `<think>` stripping, and `<<<STUDENT_SHEET>>>` / `<<<ANSWER_KEY>>>` extraction on:
1. **Indonesian SMP** prompt (4 options: `A`–`D`)
2. **Indonesian Kuliah** prompt (5 options: `A`–`E` + Essay)

In [ ]:
import re
from unsloth import FastLanguageModel

CANONICAL_SYSTEM_PROMPT = """You are an expert exam designer. Create rigorous exams matching the requested topic, audience level, format, count, and language (Indonesian or English).

Rules:
- MCQ counts: 4 options (A-D) for SD/SMP; 5 options (A-E) for SMA/SMK/Kuliah/University.
- Stems & stimuli: Self-contained with all constants/data. Ground C3-C6 items in authentic stimuli (data, code, scenarios). Bold and capitalize negative markers (**BUKAN**, **KECUALI**, **NOT**, **EXCEPT**); no double negatives.
- Distractors: Misconception-driven, same-scope, length- and tone-calibrated. No "All/None of the above" or "Semua benar/salah". Order numeric/sequential options monotonically.
- Complex MCQ & Essay/Case: Complex MCQs need independently verifiable statements and an explicit scoring/selection rule. Essays/cases need concrete tasks (predict, trace, diagnose, fix, defend trade-offs) with per-part point weights (e.g., [10 poin] or [10 pts]).
- Audit: Solve from scratch, balance MCQ key letters, and exclude answers, hints, and Bloom labels from the student sheet unless requested.

Output ONLY these two tagged sections:
<<<STUDENT_SHEET>>>
Header, instructions, and numbered questions only.
<<</STUDENT_SHEET>>>
<<<ANSWER_KEY>>>
MCQ: correct letter, why right, and one line per distractor naming its misconception. Essay/Case: model solution and point breakdown summing to each sub-part weight.
<<</ANSWER_KEY>>>"""

_THINK_RE = re.compile(r"<think>.*?</think>\s*", re.DOTALL | re.IGNORECASE)
_STUDENT_RE = re.compile(r"<<<STUDENT_SHEET>>>(.*?)<<</STUDENT_SHEET>>>", re.DOTALL)
_ANSWER_RE = re.compile(r"<<<ANSWER_KEY>>>(.*?)<<</ANSWER_KEY>>>", re.DOTALL)


def strip_thinking(text: str) -> str:
    cleaned = _THINK_RE.sub("", text or "")
    cleaned = re.sub(r"</?think>", "", cleaned, flags=re.IGNORECASE)
    return cleaned.strip()


def parse_exam_output(text: str) -> dict[str, str]:
    raw_clean = strip_thinking(text)
    s_match = _STUDENT_RE.search(raw_clean)
    a_match = _ANSWER_RE.search(raw_clean)
    return {
        "student_sheet": s_match.group(1).strip() if s_match else "",
        "answer_key": a_match.group(1).strip() if a_match else "",
        "raw_clean": raw_clean,
    }


FastLanguageModel.for_inference(model)


def generate_exam(user_prompt: str, max_new_tokens: int = 2048) -> dict[str, str]:
    messages = [
        {"role": "system", "content": CANONICAL_SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ]
    prompt_text = apply_chat_template_safe(
        messages,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    inputs = raw_tokenizer(prompt_text, return_tensors="pt").to(model.device)
    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.6,
            top_p=0.85,
            top_k=20,
            repetition_penalty=1.05,
            pad_token_id=raw_tokenizer.eos_token_id,
        )
    gen_ids = outputs[0][inputs["input_ids"].shape[-1]:]
    decoded = raw_tokenizer.decode(gen_ids, skip_special_tokens=True)
    return parse_exam_output(decoded)


# Test 1: Indonesian SMP (4-option MCQ: A-D)
smp_prompt = (
    "Buatkan paket soal ujian dengan spesifikasi berikut:\n"
    "- Topik: Tekanan Hidrostatis dan Hukum Archimedes\n"
    "- Jenjang Pendidikan: SMP\n"
    "- Profil Kognitif (Bloom): C1-C4\n"
    "- Format Soal: MCQ\n"
    "- Jumlah Soal: 4\n"
    "- Bahasa: Bahasa Indonesia"
)

print("=== TEST 1: Indonesian SMP (Expected 4 options A-D) ===")
smp_res = generate_exam(smp_prompt)
print(f"Extracted student_sheet chars: {len(smp_res['student_sheet'])} | answer_key chars: {len(smp_res['answer_key'])}")
assert smp_res["student_sheet"] and smp_res["answer_key"], "Failed to extract SMP tagged sections!"
print("\n--- SMP STUDENT SHEET ---\n", smp_res["student_sheet"][:1500])
print("\n--- SMP ANSWER KEY ---\n", smp_res["answer_key"][:1500])

In [ ]:
# Test 2: Indonesian Kuliah (5-option MCQ A-E + Essay)
kuliah_prompt = (
    "Buatkan paket soal ujian dengan spesifikasi berikut:\n"
    "- Topik: Algoritma Pencarian Heuristik (A* dan Greedy Best-First Search)\n"
    "- Jenjang Pendidikan: Kuliah\n"
    "- Profil Kognitif (Bloom): C3-C6\n"
    "- Format Soal: Mixed (2 MCQ, 1 Essay)\n"
    "- Jumlah Soal: 3\n"
    "- Bahasa: Bahasa Indonesia"
)

print("=== TEST 2: Indonesian Kuliah (Expected 5 options A-E + Essay) ===")
kuliah_res = generate_exam(kuliah_prompt)
print(f"Extracted student_sheet chars: {len(kuliah_res['student_sheet'])} | answer_key chars: {len(kuliah_res['answer_key'])}")
assert kuliah_res["student_sheet"] and kuliah_res["answer_key"], "Failed to extract Kuliah tagged sections!"
print("\n--- KULIAH STUDENT SHEET ---\n", kuliah_res["student_sheet"][:1800])
print("\n--- KULIAH ANSWER KEY ---\n", kuliah_res["answer_key"][:1800])

## Step 6 — Export LoRA, GGUF (`q4_k_m`), and Ollama `Modelfile`

Saves:
1. LoRA adapter weights (`qwen3.5_2b_exam_lora/`)
2. Merged & quantized `Q4_K_M` GGUF (`qwen3.5_2b_exam_gguf/`)
3. Auto-generated Ollama `Modelfile` matched to the exact Qwen3.5 ChatML template used during training.

In [ ]:
import glob
from pathlib import Path

LORA_DIR = "qwen3.5_2b_exam_lora"
GGUF_DIR = "qwen3.5_2b_exam_gguf"

# 1. Save LoRA adapters
model.save_pretrained(LORA_DIR)
tokenizer.save_pretrained(LORA_DIR)
print(f"Saved LoRA adapter to ./{LORA_DIR}/")

# 2. Export Q4_K_M GGUF (~1.5 GB — fits RTX 3050 4GB and RTX 5060 Ti 16GB)
model.save_pretrained_gguf(
    GGUF_DIR,
    tokenizer,
    quantization_method="q4_k_m",
)

gguf_files = sorted(glob.glob(f"{GGUF_DIR}/**/*.gguf", recursive=True))
print("Exported GGUF files:", gguf_files)
gguf_rel_path = gguf_files[0] if gguf_files else f"./{GGUF_DIR}/Qwen3.5-2B.Q4_K_M.gguf"
if not gguf_rel_path.startswith("./") and not gguf_rel_path.startswith("/"):
    gguf_rel_path = "./" + gguf_rel_path.replace("\\", "/")

# 3. Generate Modelfile matching training response_part
think_prefill = "<think>\n\n</think>\n\n" if "<think>" in response_part else ""

MODELFILE_TEMPLATE = """FROM __GGUF_PATH__

TEMPLATE \"\"\"{{- if .Messages }}{{- if .System }}<|im_start|>system
{{ .System }}<|im_end|>
{{ end }}{{- range $i, $_ := .Messages }}{{- $last := eq (len (slice $.Messages $i)) 1 -}}{{- if eq .Role "user" }}<|im_start|>user
{{ .Content }}<|im_end|>
{{ else if eq .Role "assistant" }}<|im_start|>assistant
__THINK_PREFILL__{{ .Content }}{{ if not $last }}<|im_end|>
{{ end }}{{- end }}{{- if and (ne .Role "assistant") $last }}<|im_start|>assistant
__THINK_PREFILL__{{ end }}{{- end }}{{- else }}{{- if .System }}<|im_start|>system
{{ .System }}<|im_end|>
{{ end }}{{- if .Prompt }}<|im_start|>user
{{ .Prompt }}<|im_end|>
{{ end }}<|im_start|>assistant
__THINK_PREFILL__{{ end }}{{ .Response }}{{ if .Response }}<|im_end|>{{ end }}\"\"\"

SYSTEM \"\"\"__SYSTEM_PROMPT__\"\"\"

PARAMETER stop "<|im_start|>"
PARAMETER stop "<|im_end|>"
PARAMETER stop "<|endoftext|>"
PARAMETER num_ctx 4096
PARAMETER temperature 0.6
PARAMETER top_p 0.85
PARAMETER top_k 20
PARAMETER repeat_penalty 1.05
"""

modelfile_content = (
    MODELFILE_TEMPLATE
    .replace("__GGUF_PATH__", gguf_rel_path)
    .replace("__THINK_PREFILL__", think_prefill)
    .replace("__SYSTEM_PROMPT__", CANONICAL_SYSTEM_PROMPT)
)

Path("Modelfile").write_text(modelfile_content, encoding="utf-8")
print("Generated ./Modelfile pointing to:", gguf_rel_path)
print("\n--- Modelfile Preview (first 25 lines) ---")
print("\n".join(modelfile_content.splitlines()[:25]))

In [ ]:
# Optional: Copy artifacts to Google Drive for local download
SAVE_TO_DRIVE = False
DRIVE_EXPORT_DIR = Path("/content/drive/MyDrive/qwen_exam_generator_export")

if SAVE_TO_DRIVE and "google.colab" in sys.modules:
    import shutil
    from google.colab import drive

    drive.mount("/content/drive")
    DRIVE_EXPORT_DIR.mkdir(parents=True, exist_ok=True)
    shutil.copy2("Modelfile", DRIVE_EXPORT_DIR / "Modelfile")
    shutil.copytree(LORA_DIR, DRIVE_EXPORT_DIR / LORA_DIR, dirs_exist_ok=True)
    shutil.copytree(GGUF_DIR, DRIVE_EXPORT_DIR / GGUF_DIR, dirs_exist_ok=True)
    print(f"Copied LoRA, GGUF, and Modelfile to {DRIVE_EXPORT_DIR}")
else:
    print("Tip: Set SAVE_TO_DRIVE = True to back up your GGUF & Modelfile to Google Drive,")
    print("or download qwen3.5_2b_exam_gguf/ + Modelfile directly from the Colab Files pane.")

### Local Deployment Instructions (RTX 3050 4GB / RTX 5060 Ti 16GB)

1. Download `Modelfile` and the `.gguf` file inside `qwen3.5_2b_exam_gguf/` into your local project folder.
2. Adjust the `FROM ...` path in `Modelfile` if you moved the `.gguf` file (or regenerate it with `python local_runner.py --write-modelfile <path-to-gguf>`).
3. Register and run in **Ollama**:
   ```bash
   ollama create qwen3.5-exam -f Modelfile
   ollama run qwen3.5-exam
   ```
4. Test and validate against `schema.py` / `validate_dataset.py` using `local_runner.py`:
   ```bash
   python local_runner.py --backend ollama --model qwen3.5-exam --topic "Basis Data Relasional" --level Kuliah --formats Mixed --num 4 --validate
   ```